# Paper 6: Coupling-Channel Structure Predicts Cancer Survival Beyond Mutation Count

**Reproducibility notebook** for McCarthy (2026), Paper 6.

Validates the coupling-channel hypothesis on the MSK-MET 2021 metastatic cohort (25,776 patients).
Shows that (1) channel count resolves the TMB paradox, (2) channels pair into four symmetric tiers
whose logic gates all add signal, and (3) cross-channel breadth matters more than within-channel depth.

All data fetched live from the public [cBioPortal API](https://www.cbioportal.org/api).
No GPU, Neo4j, or local files required. Runtime: ~5 minutes (API-bound).

**Dataset:** MSK-MET 2021 — MetTropism Cohort (Nguyen et al. 2022)

In [ ]:
# Cell 1: Setup
!pip install -q lifelines pandas numpy matplotlib requests scipy

import os
import json
import time as _time
import warnings
import requests
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from scipy.stats import wilcoxon
%matplotlib inline

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10
print("Setup complete.")

In [ ]:
# cBioPortal API helpers
import time as _time

BASE_URL = "https://www.cbioportal.org/api"
HEADERS = {"Accept": "application/json"}


def api_get(path, params=None, retries=3):
    for attempt in range(retries):
        try:
            resp = requests.get(f"{BASE_URL}{path}", headers=HEADERS, params=params, timeout=60)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            if attempt < retries - 1:
                _time.sleep(5 * (attempt + 1))
            else:
                raise


def api_post(path, body, retries=3):
    for attempt in range(retries):
        try:
            resp = requests.post(
                f"{BASE_URL}{path}", json=body,
                headers={**HEADERS, "Content-Type": "application/json"}, timeout=60
            )
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            if attempt < retries - 1:
                _time.sleep(5 * (attempt + 1))
            else:
                raise

In [ ]:
# Cell 3: Channel mapping — 122 genes across 8 coupling channels

CHANNEL_MAP = {
    # DDR — DNA Damage Response
    "ATM": "DDR", "ATR": "DDR", "BRCA1": "DDR", "BRCA2": "DDR",
    "PALB2": "DDR", "RAD51C": "DDR", "RAD51D": "DDR", "RAD51B": "DDR",
    "CHEK1": "DDR", "CHEK2": "DDR", "BAP1": "DDR", "BARD1": "DDR",
    "FANCA": "DDR", "FANCC": "DDR", "FANCD2": "DDR", "MLH1": "DDR",
    "MSH2": "DDR", "MSH6": "DDR", "PMS2": "DDR", "POLE": "DDR",
    "POLD1": "DDR",

    # CellCycle — Cell Cycle / Apoptosis
    "TP53": "CellCycle", "RB1": "CellCycle", "CDKN1A": "CellCycle", "CDKN1B": "CellCycle",
    "CDKN2A": "CellCycle", "CDKN2B": "CellCycle", "CDK4": "CellCycle", "CDK6": "CellCycle",
    "CCND1": "CellCycle", "CCNE1": "CellCycle", "MDM2": "CellCycle", "MDM4": "CellCycle",
    "MYC": "CellCycle", "MYCN": "CellCycle",

    # PI3K_Growth — PI3K / Growth Signaling
    "PIK3CA": "PI3K_Growth", "PIK3R1": "PI3K_Growth", "PTEN": "PI3K_Growth", "AKT1": "PI3K_Growth",
    "AKT2": "PI3K_Growth", "AKT3": "PI3K_Growth", "MTOR": "PI3K_Growth", "KRAS": "PI3K_Growth",
    "NRAS": "PI3K_Growth", "HRAS": "PI3K_Growth", "BRAF": "PI3K_Growth", "RAF1": "PI3K_Growth",
    "MAP2K1": "PI3K_Growth", "MAP2K2": "PI3K_Growth", "MAP3K1": "PI3K_Growth", "MAP3K13": "PI3K_Growth",
    "ERBB2": "PI3K_Growth", "ERBB3": "PI3K_Growth", "EGFR": "PI3K_Growth", "FGFR1": "PI3K_Growth",
    "FGFR2": "PI3K_Growth", "FGFR3": "PI3K_Growth", "IGF1R": "PI3K_Growth", "MET": "PI3K_Growth",
    "NF1": "PI3K_Growth", "NF2": "PI3K_Growth", "TSC1": "PI3K_Growth", "TSC2": "PI3K_Growth",
    "STK11": "PI3K_Growth",

    # Endocrine — Endocrine / Hormone Receptor
    "ESR1": "Endocrine", "ESR2": "Endocrine", "PGR": "Endocrine", "AR": "Endocrine",
    "FOXA1": "Endocrine", "GATA3": "Endocrine",

    # Immune — Immune Surveillance
    "B2M": "Immune", "HLA-A": "Immune", "HLA-B": "Immune", "HLA-C": "Immune",
    "JAK1": "Immune", "JAK2": "Immune", "STAT1": "Immune", "CD274": "Immune",
    "PDCD1LG2": "Immune", "CTLA4": "Immune",

    # TissueArchitecture — Tissue Architecture
    "CDH1": "TissueArchitecture", "CDH2": "TissueArchitecture", "CTNNB1": "TissueArchitecture", "APC": "TissueArchitecture",
    "AXIN1": "TissueArchitecture", "AXIN2": "TissueArchitecture", "SMAD2": "TissueArchitecture", "SMAD3": "TissueArchitecture",
    "SMAD4": "TissueArchitecture", "TGFBR1": "TissueArchitecture", "TGFBR2": "TissueArchitecture", "NOTCH1": "TissueArchitecture",
    "NOTCH2": "TissueArchitecture", "NOTCH3": "TissueArchitecture", "NOTCH4": "TissueArchitecture", "FBXW7": "TissueArchitecture",
    "GJA1": "TissueArchitecture", "GJB2": "TissueArchitecture",

    # ChromatinRemodel — Chromatin Remodeling
    "KMT2D": "ChromatinRemodel", "KMT2C": "ChromatinRemodel", "KMT2A": "ChromatinRemodel", "KMT2B": "ChromatinRemodel",
    "SETD2": "ChromatinRemodel", "NSD1": "ChromatinRemodel", "CREBBP": "ChromatinRemodel", "EP300": "ChromatinRemodel",
    "ARID1A": "ChromatinRemodel", "ARID1B": "ChromatinRemodel", "ARID2": "ChromatinRemodel", "SMARCA4": "ChromatinRemodel",
    "KDM6A": "ChromatinRemodel", "BCOR": "ChromatinRemodel", "H3C7": "ChromatinRemodel", "ANKRD11": "ChromatinRemodel",

    # DNAMethylation — DNA Methylation
    "DNMT3A": "DNAMethylation", "DNMT3B": "DNAMethylation", "TET1": "DNAMethylation", "TET2": "DNAMethylation",
    "IDH1": "DNAMethylation", "IDH2": "DNAMethylation", "ATRX": "DNAMethylation", "DAXX": "DNAMethylation",

}

CHANNEL_NAMES = {
    "DDR": "DNA Damage Response",
    "CellCycle": "Cell Cycle / Apoptosis",
    "PI3K_Growth": "PI3K / Growth Signaling",
    "Endocrine": "Endocrine / Hormone",
    "Immune": "Immune Surveillance",
    "TissueArchitecture": "Tissue Architecture",
    "ChromatinRemodel": "Chromatin Remodeling",
    "DNAMethylation": "DNA Methylation",
}

CHANNEL_COLORS = {
    "DDR": "#e74c3c", "CellCycle": "#e67e22",
    "PI3K_Growth": "#f1c40f", "Endocrine": "#2ecc71",
    "Immune": "#3498db", "TissueArchitecture": "#9b59b6",
    "ChromatinRemodel": "#1abc9c", "DNAMethylation": "#e84393",
}

DRIVER_MUTATION_TYPES = {
    "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
    "Frame_Shift_Ins", "Splice_Site", "In_Frame_Del", "In_Frame_Ins",
    "Nonstop_Mutation", "Translation_Start_Site",
}

# Tier structure: 4 tiers × 2 channels each
TIER_STRUCTURE = {
    0: {"name": "Cell-Intrinsic", "channels": ["CellCycle", "PI3K_Growth"]},
    1: {"name": "Microenvironment", "channels": ["TissueArchitecture", "Immune"]},
    2: {"name": "Organism", "channels": ["Endocrine", "DDR"]},
    3: {"name": "Meta/Epigenetic", "channels": ["ChromatinRemodel", "DNAMethylation"]},
}

print(f"{len(CHANNEL_MAP)} genes mapped to {len(CHANNEL_NAMES)} channels, {len(TIER_STRUCTURE)} tiers.")

In [ ]:
STUDY_ID = "msk_met_2021"
CACHE = "/content/"
os.makedirs(CACHE, exist_ok=True)


def fetch_clinical(study_id):
    path = os.path.join(CACHE, f"{study_id}_clinical.csv")
    if os.path.exists(path):
        print(f"  Loading cached clinical data...")
        return pd.read_csv(path, index_col=0)
    print(f"  Fetching clinical data for {study_id}...")
    data = api_get(f"/studies/{study_id}/clinical-data",
                   {"clinicalDataType": "PATIENT", "projection": "DETAILED"})
    df = pd.DataFrame(data)
    pivot = df.pivot_table(index="patientId", columns="clinicalAttributeId",
                           values="value", aggfunc="first")
    pivot.to_csv(path)
    print(f"  Cached {len(pivot)} patients.")
    return pivot


def fetch_mutations(study_id):
    path = os.path.join(CACHE, f"{study_id}_mutations.csv")
    if os.path.exists(path):
        print(f"  Loading cached mutations...")
        return pd.read_csv(path)
    print(f"  Fetching mutations for {study_id}...")
    samples = api_get(f"/studies/{study_id}/samples")
    sample_ids = [s["sampleId"] for s in samples]
    profiles = api_get(f"/studies/{study_id}/molecular-profiles")
    mut_profile = next(
        (p["molecularProfileId"] for p in profiles
         if p.get("molecularAlterationType") == "MUTATION_EXTENDED"), None)
    if not mut_profile:
        raise ValueError(f"No mutation profile for {study_id}")
    all_muts = []
    batch_size = 500
    for i in range(0, len(sample_ids), batch_size):
        batch = sample_ids[i:i + batch_size]
        muts = api_post(f"/molecular-profiles/{mut_profile}/mutations/fetch",
                        {"sampleIds": batch})
        all_muts.extend(muts)
        if (i // batch_size) % 20 == 0:
            print(f"    batch {i // batch_size + 1}/{(len(sample_ids) - 1) // batch_size + 1}")
    df = pd.json_normalize(all_muts)
    if "gene.hugoGeneSymbol" not in df.columns and "keyword" in df.columns:
        df["gene.hugoGeneSymbol"] = df["keyword"].apply(
            lambda k: str(k).split()[0] if pd.notna(k) else None)
    df.to_csv(path, index=False)
    print(f"  Cached {len(df)} mutations.")
    return df


print(f"=== Fetching {STUDY_ID} ===")
clinical_df = fetch_clinical(STUDY_ID)
mut_df = fetch_mutations(STUDY_ID)
print(f"Clinical: {len(clinical_df)} patients, Mutations: {len(mut_df)} rows")

In [ ]:
# Cell 5: Build patient dataframe

gene_col = "gene.hugoGeneSymbol"

# Filter to non-silent driver mutations
driver_mut = mut_df[mut_df["mutationType"].isin(DRIVER_MUTATION_TYPES)].copy()
driver_mut["channel"] = driver_mut[gene_col].map(CHANNEL_MAP)
mapped = driver_mut.dropna(subset=["channel"])

# Per-patient aggregations
all_patients = pd.DataFrame({"patientId": mut_df["patientId"].unique()})

ch_count = mapped.groupby("patientId")["channel"].nunique().reset_index()
ch_count.columns = ["patientId", "channel_count"]

ch_set = mapped.groupby("patientId")["channel"].apply(set).reset_index()
ch_set.columns = ["patientId", "channels_severed"]

drv_count = driver_mut.groupby("patientId")[gene_col].count().reset_index()
drv_count.columns = ["patientId", "driver_mutation_count"]

tot_count = mut_df.groupby("patientId")[gene_col].count().reset_index()
tot_count.columns = ["patientId", "total_mutation_count"]

pdf = all_patients
for right in [ch_count, ch_set, drv_count, tot_count]:
    pdf = pdf.merge(right, on="patientId", how="left")
pdf["channel_count"] = pdf["channel_count"].fillna(0).astype(int)
pdf["driver_mutation_count"] = pdf["driver_mutation_count"].fillna(0).astype(int)
pdf["total_mutation_count"] = pdf["total_mutation_count"].fillna(0).astype(int)
pdf["channels_severed"] = pdf["channels_severed"].apply(lambda x: x if isinstance(x, set) else set())

# Per-channel binary flags
for ch in CHANNEL_NAMES:
    pdf[ch] = pdf["channels_severed"].apply(lambda s: int(ch in s))

# Survival
surv = clinical_df[["OS_STATUS", "OS_MONTHS"]].copy().reset_index()
surv.columns = ["patientId", "os_status", "os_months"]
surv["os_months"] = pd.to_numeric(surv["os_months"], errors="coerce")
surv["event"] = surv["os_status"].apply(lambda x: 1 if "DECEASED" in str(x) else 0)
surv = surv.dropna(subset=["os_months"])
surv = surv[surv["os_months"] > 0]

# Cancer type
if "CANCER_TYPE" in clinical_df.columns:
    ct = clinical_df[["CANCER_TYPE"]].reset_index()
    ct.columns = ["patientId", "cancer_type"]
    surv = surv.merge(ct, on="patientId", how="left")

df = pdf.merge(surv, on="patientId", how="inner")
print(f"Final: {len(df)} patients, {int(df['event'].sum())} deaths")
print(f"Median follow-up: {df['os_months'].median():.1f} months")

In [ ]:
# Cell 6: Table 1 — Channel frequency and mortality by channel count

print("=== CHANNEL FREQUENCY ===\n")
for ch, name in CHANNEL_NAMES.items():
    n = df[ch].sum()
    print(f"  {name:30s}: {n:5d} ({100*n/len(df):.1f}%)")

print(f"\n=== MORTALITY BY CHANNEL COUNT ===\n")
for n_ch in sorted(df["channel_count"].unique()):
    sub = df[df["channel_count"] == n_ch]
    d = sub["event"].sum()
    print(f"  {n_ch} channels: {len(sub):5d} pts, {int(d):4d} deaths ({100*d/len(sub):5.1f}%)")

In [ ]:
# Cell 7: Table 2 — TMB Paradox Resolution (Cox regression)

cox_df = df[["os_months", "event", "channel_count", "driver_mutation_count"]].dropna().copy()
cox_df["log_mut"] = np.log1p(cox_df["driver_mutation_count"])

print("=== TABLE 2: TMB PARADOX RESOLUTION ===\n")

models = [
    ("channel_count only", ["channel_count"]),
    ("log_mut only", ["log_mut"]),
    ("both (multivariate)", ["channel_count", "log_mut"]),
]

for name, cols in models:
    cph = CoxPHFitter()
    cph.fit(cox_df[["os_months", "event"] + cols], duration_col="os_months", event_col="event")
    print(f"--- {name} ---")
    for var in cols:
        r = cph.summary.loc[var]
        print(f"  {var:20s}: HR={r['exp(coef)']:.4f} "
              f"(95% CI {r['exp(coef) lower 95%']:.4f}–{r['exp(coef) upper 95%']:.4f}), "
              f"p={r['p']:.2e}")
    print(f"  C-index: {cph.concordance_index_:.4f}")
    print()

In [ ]:
# Cell 8: Figure 1 — KM curves by channel count vs mutation count

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel A: Channel count
ax = axes[0]
kmf = KaplanMeierFitter()
groups = [
    ("0 channels", df["channel_count"] == 0, "#3498db"),
    ("1 channel",  df["channel_count"] == 1, "#2ecc71"),
    ("2 channels", df["channel_count"] == 2, "#f39c12"),
    ("3+ channels", df["channel_count"] >= 3, "#e74c3c"),
]
for label, mask, color in groups:
    sub = df[mask]
    if len(sub) < 10:
        continue
    kmf.fit(sub["os_months"], sub["event"], label=f"{label} (n={len(sub)})")
    kmf.plot_survival_function(ax=ax, ci_show=True, color=color)

low = df[df["channel_count"] <= 1]
high = df[df["channel_count"] >= 3]
if len(low) > 10 and len(high) > 10:
    lr = logrank_test(low["os_months"], high["os_months"], low["event"], high["event"])
    ax.text(0.95, 0.05, f"0-1 vs 3+: p={lr.p_value:.2e}",
            transform=ax.transAxes, ha="right", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax.set_xlabel("Months")
ax.set_ylabel("Overall Survival")
ax.set_title("A. By Coupling Channels Severed\nMSK-MET 2021", fontweight="bold")
ax.legend(fontsize=9, loc="lower left")
ax.set_ylim(0, 1.05)

# Panel B: Mutation count
ax = axes[1]
kmf = KaplanMeierFitter()
q33 = df["driver_mutation_count"].quantile(0.33)
q67 = df["driver_mutation_count"].quantile(0.67)
if q33 == q67:
    q33, q67 = df["driver_mutation_count"].median() - 1, df["driver_mutation_count"].median() + 1

for label, mask, color in [
    (f"Low (\u2264{int(q33)})", df["driver_mutation_count"] <= q33, "#2ecc71"),
    (f"Med ({int(q33)+1}-{int(q67)})", (df["driver_mutation_count"] > q33) & (df["driver_mutation_count"] <= q67), "#f39c12"),
    (f"High (>{int(q67)})", df["driver_mutation_count"] > q67, "#e74c3c"),
]:
    sub = df[mask]
    if len(sub) < 10:
        continue
    kmf.fit(sub["os_months"], sub["event"], label=f"{label} (n={len(sub)})")
    kmf.plot_survival_function(ax=ax, ci_show=True, color=color)

ax.set_xlabel("Months")
ax.set_ylabel("Overall Survival")
ax.set_title("B. By Non-Silent Mutation Count\nMSK-MET 2021", fontweight="bold")
ax.legend(fontsize=9, loc="lower left")
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Cross-channel breadth vs same-channel depth

cross = df[df["channel_count"] >= 2]
same = df[(df["driver_mutation_count"] >= 2) & (df["channel_count"] == 1)]

print(f"Cross-channel (2+ channels): {len(cross)} pts, "
      f"{100*cross['event'].mean():.1f}% mortality")
print(f"Same-channel (2+ muts, 1 ch): {len(same)} pts, "
      f"{100*same['event'].mean():.1f}% mortality")

if len(cross) > 10 and len(same) > 10:
    lr = logrank_test(cross["os_months"], same["os_months"],
                      cross["event"], same["event"])
    print(f"Log-rank p: {lr.p_value:.2e}")

    fig, ax = plt.subplots(figsize=(8, 5))
    kmf = KaplanMeierFitter()
    kmf.fit(cross["os_months"], cross["event"],
            label=f"Cross-channel, 2+ channels (n={len(cross)})")
    kmf.plot_survival_function(ax=ax, ci_show=True, color="#e74c3c")
    kmf.fit(same["os_months"], same["event"],
            label=f"Same-channel, 2+ mutations (n={len(same)})")
    kmf.plot_survival_function(ax=ax, ci_show=True, color="#3498db")
    ax.set_xlabel("Months")
    ax.set_ylabel("Overall Survival")
    ax.set_title("Cross-channel breadth vs same-channel depth\nMSK-MET 2021", fontweight="bold")
    ax.legend(fontsize=9, loc="lower left")
    ax.text(0.95, 0.05, f"Log-rank p={lr.p_value:.2e}",
            transform=ax.transAxes, ha="right", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 10: Tier logic gates — NOR/AND/XOR/OR across cancer types

# Build tier indicators per patient
for tier_id, tier in TIER_STRUCTURE.items():
    ch_a, ch_b = tier["channels"]
    df[f"tier{tier_id}_NOR"] = ((df[ch_a] == 0) & (df[ch_b] == 0)).astype(int)
    df[f"tier{tier_id}_AND"] = ((df[ch_a] == 1) & (df[ch_b] == 1)).astype(int)
    df[f"tier{tier_id}_XOR"] = ((df[ch_a] != df[ch_b])).astype(int)
    df[f"tier{tier_id}_OR"]  = ((df[ch_a] == 1) | (df[ch_b] == 1)).astype(int)

ch_cols = list(CHANNEL_NAMES.keys())

# Per-cancer-type comparison
if "cancer_type" in df.columns:
    ct_counts = df["cancer_type"].value_counts()
    eligible_cts = ct_counts[(ct_counts >= 50)].index.tolist()
    # Further filter: need >=10 events
    eligible_cts = [ct for ct in eligible_cts
                    if df[df["cancer_type"] == ct]["event"].sum() >= 10]
    print(f"Eligible cancer types (>=50 pts, >=10 events): {len(eligible_cts)}\n")

    results = {}
    for gate in ["NOR", "AND", "XOR", "OR"]:
        gate_cols = [f"tier{t}_{gate}" for t in range(4)]
        wins, losses, ties = 0, 0, 0
        deltas = []

        for ct in eligible_cts:
            ct_df = df[df["cancer_type"] == ct][["os_months", "event"] + ch_cols + gate_cols].dropna()
            if len(ct_df) < 50 or ct_df["event"].sum() < 10:
                continue
            # Check variance
            base_cols_use = [c for c in ch_cols if ct_df[c].std() > 0]
            gate_cols_use = [c for c in gate_cols if ct_df[c].std() > 0]
            if len(base_cols_use) < 2 or len(gate_cols_use) == 0:
                continue

            try:
                cph_base = CoxPHFitter(penalizer=0.1)
                cph_base.fit(ct_df[["os_months", "event"] + base_cols_use],
                             duration_col="os_months", event_col="event")
                c_base = cph_base.concordance_index_

                cph_aug = CoxPHFitter(penalizer=0.1)
                cph_aug.fit(ct_df[["os_months", "event"] + base_cols_use + gate_cols_use],
                            duration_col="os_months", event_col="event")
                c_aug = cph_aug.concordance_index_

                delta = c_aug - c_base
                deltas.append(delta)
                if delta > 0.001:
                    wins += 1
                elif delta < -0.001:
                    losses += 1
                else:
                    ties += 1
            except Exception:
                continue

        if len(deltas) >= 5:
            try:
                stat, p = wilcoxon(deltas, alternative="greater")
            except Exception:
                stat, p = 0, 1.0
        else:
            stat, p = 0, 1.0

        results[gate] = {"wins": wins, "losses": losses, "ties": ties,
                         "mean_delta": np.mean(deltas) if deltas else 0,
                         "p": p, "n": len(deltas)}

    print(f"{'Gate':>6s}  {'Wins':>5s}  {'Loss':>5s}  {'Ties':>5s}  {'Mean \u0394C':>10s}  {'Wilcoxon p':>12s}  {'N':>3s}")
    print("-" * 55)
    for gate, r in results.items():
        print(f"  {gate:>4s}  {r['wins']:>5d}  {r['losses']:>5d}  {r['ties']:>5d}  "
              f"{r['mean_delta']:>10.4f}  {r['p']:>12.2e}  {r['n']:>3d}")
else:
    print("Cancer type not available — skipping per-type tier analysis.")

In [ ]:
# Cell 11: Figure 2 — Channel co-occurrence heatmap

ch_list = list(CHANNEL_NAMES.keys())
n_ch = len(ch_list)
cooccurrence = np.zeros((n_ch, n_ch))

for i, ch_a in enumerate(ch_list):
    for j, ch_b in enumerate(ch_list):
        if i == j:
            cooccurrence[i, j] = 1.0
        else:
            either = df[(df[ch_a] == 1) | (df[ch_b] == 1)]
            if len(either) > 0:
                both = ((either[ch_a] == 1) & (either[ch_b] == 1)).sum()
                cooccurrence[i, j] = both / len(either)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cooccurrence, cmap="YlOrRd", vmin=0, vmax=0.5)

short_names = [CHANNEL_NAMES[ch].split("/")[0].split("(")[0].strip()[:12] for ch in ch_list]
ax.set_xticks(range(n_ch))
ax.set_xticklabels(short_names, rotation=45, ha="right")
ax.set_yticks(range(n_ch))
ax.set_yticklabels(short_names)

for i in range(n_ch):
    for j in range(n_ch):
        if i != j:
            ax.text(j, i, f"{cooccurrence[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax, label="P(both | either)")
ax.set_title("Channel Co-occurrence\nP(both severed | either severed)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Per-channel KM curves

fig, ax = plt.subplots(figsize=(10, 7))
kmf = KaplanMeierFitter()

for channel, name in CHANNEL_NAMES.items():
    has = df[channel] == 1
    sub = df[has]
    if len(sub) < 15:
        continue
    kmf.fit(sub["os_months"], sub["event"],
            label=f"{name} (n={len(sub)})")
    kmf.plot_survival_function(ax=ax, color=CHANNEL_COLORS.get(channel, "gray"))

no_ch = df[df["channel_count"] == 0]
if len(no_ch) >= 15:
    kmf.fit(no_ch["os_months"], no_ch["event"],
            label=f"No channel mutations (n={len(no_ch)})")
    kmf.plot_survival_function(ax=ax, color="gray", linestyle="--")

ax.set_xlabel("Months")
ax.set_ylabel("Overall Survival")
ax.set_title("Per-Channel Survival\nMSK-MET 2021", fontweight="bold")
ax.legend(fontsize=8, loc="lower left")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()